# Assignment 16 — Financial Portfolio Manager (Group Chat + StateFlow, AG2)

**Objective:** Build a Financial Portfolio Manager that helps a user manage their investments using **Group Chat** for agent collaboration and **StateFlow** for dynamic workflow management. A **User Proxy Agent** initiates a chat with the **Group Chat Manager**, which calls the **Portfolio Analysis Agent** (summarizes the portfolio and picks *Growth* or *Value*). **StateFlow** then routes the conversation to either the **Growth Investment Agent** or the **Value Investment Agent**, and finally the **Investment Advisor Agent** compiles a personalized financial report, which is displayed as the final output.

**Framework:** AG2 `1.1.2` (`pip install ag2[openai]`), the current AG2 release. Since AG2 v1.0 the top-level package is the protocol-driven API (`from ag2 import Agent`); the classic `autogen.GroupChat` / `GroupChatManager` classes live in the separate *AG2 Classic* code base, and the 1.x native equivalent is the opt-in `ag2.network` module (`Hub` + `WorkflowAdapter` + `TransitionGraph`). To keep this assignment simple and transparent, the group chat and StateFlow are implemented here in a few lines of plain Python on top of AG2 agents, exactly as the course teaches them:

| Assignment concept | AG2 Classic (course handouts) | This notebook (AG2 1.x) |
|---|---|---|
| LLM agent with a role | `ConversableAgent(system_message=...)` | `Agent(name, prompt=..., config=OpenAIConfig(...))` |
| Group Chat (shared conversation) | `GroupChat(agents, messages, max_round, speaker_selection_method)` | `GroupChat` class: shared `messages` list that every speaker sees in full |
| Group Chat Manager | `GroupChatManager(groupchat)` | `GroupChatManager` class: runs rounds, asks StateFlow who speaks next, stops at `None` or `max_round` |
| StateFlow (dynamic workflow) | `speaker_selection_method=state_transition(last_speaker, groupchat)` | Same function signature and logic: a deterministic state machine over the last speaker + last message |
| User Proxy initiates the chat | `user_proxy.initiate_chat(group_chat_manager, message=...)` | `UserProxyAgent.initiate_chat(manager, message)` |

### How to run (Google Colab)
1. **Colab Secrets (recommended):** click the 🔑 icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and turn on *Notebook access*.
2. **No secret?** Just run the notebook. If no key is found you will be prompted to paste one (hidden input, never printed or saved). An `OPENAI_API_KEY` environment variable also works.
3. `Runtime → Run all`. In the data-collection cell, press **Enter** to accept each default or type your own salary and portfolio summary. The last cell prints a PASS/FAIL rubric.

> This is a course exercise. The generated report is general educational content produced by an LLM, not licensed financial advice.

### Rubric (verified automatically in the last cell)

| # | Expectation (from the assignment) | Where it is satisfied |
|---|---|---|
| R1 | **User Proxy Agent collects** the inputs: current salary + summary of the investment portfolio (validated; bad input rejected) | User Proxy cell + self-test |
| R2 | User Proxy **initiates the chat with the Group Chat Manager** and states the intention to manage investments | Run cell (first group-chat message) |
| R3 | Group Chat Manager **calls the Portfolio Analysis Agent** first | Manager + StateFlow (round 1) |
| R4 | Portfolio Analysis Agent **summarizes the portfolio and determines Growth or Value** | Portfolio agent prompt + parsed `CATEGORY:` line |
| R5 | **StateFlow dynamically routes** to the Growth *or* the Value Investment Agent based on that category (the other never speaks) | StateFlow cell + speaker order |
| R6 | StateFlow is a **deterministic** state machine (both branches + fallback unit-tested without an LLM) | StateFlow self-test |
| R7 | Growth / Value Investment Agent **provides specific investment recommendations** | Growth / Value agent output |
| R8 | Investment Advisor Agent speaks **last** and **compiles a detailed report** from the previous agents' insights | Advisor prompt + speaker order |
| R9 | Conversation **ends through StateFlow** (returns `None`) within `max_round` | Manager log |
| R10 | Agents **collaborate through the group chat**: every speaker sees all previous messages | Manager log (`visible_messages`) |
| R11 | The **final report includes portfolio analysis and investment suggestions** (required sections present) | Final report cell |
| R12 | **Output of the Investment Advisor Agent is displayed** (Markdown) | Final report cell |


## 1. Install dependencies
Installed with **uv** (fast). Versions are pinned so the notebook stays reproducible.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python "{sys.executable}" "ag2[openai]==1.1.2"

import ag2
print("AG2 version:", ag2.__version__)

## 2. API key
Looks for the key in **Colab Secrets** first, then an environment variable, then asks you to paste it. The key is never printed.

In [ ]:
import os
from getpass import getpass


def load_api_key() -> str:
    try:
        from google.colab import userdata  # only exists inside Colab
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not in Colab, secret missing, or notebook access not granted
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    return getpass("OPENAI_API_KEY not found in Colab Secrets or environment. Paste your key: ")


API_KEY = load_api_key().strip()
if not API_KEY:
    raise ValueError("No API key provided. Add OPENAI_API_KEY to Colab Secrets or paste it when prompted.")
print("API key loaded.")

## 3. Settings and model
All settings live here. The first reachable model in `MODEL_CANDIDATES` is used. Set `INTERACTIVE_INPUT = False` to skip the prompts and use `DEFAULT_INPUTS` (useful for unattended runs).

In [ ]:
from ag2 import Agent
from ag2.config import OpenAIConfig

MODEL_CANDIDATES = ["gpt-5.4-mini", "gpt-5-mini", "gpt-4.1-mini"]  # tried in order; first reachable wins
MAX_ROUND = 6  # safety cap on group-chat rounds; StateFlow normally ends the chat after 4

INTERACTIVE_INPUT = True  # True: prompt for each value (Enter accepts the default). False: use DEFAULT_INPUTS as-is.
DEFAULT_INPUTS = {
    "salary": 95000,
    "currency": "USD",
    "portfolio_summary": (
        "Fixed deposits: 40,000; SIPs in index mutual funds: 25,000 (adding 500/month); "
        "Real estate: one apartment worth 150,000 with 90,000 mortgage remaining; "
        "Employer retirement account: 60,000; Emergency fund in savings: 15,000; No individual stocks."
    ),
}


async def pick_model() -> str:
    """Fail fast with a clear message if the key or every model name is wrong."""
    for model in MODEL_CANDIDATES:
        try:
            probe = Agent("probe", prompt="Reply with the single word: OK", config=OpenAIConfig(model=model, api_key=API_KEY))
            reply = await probe.ask("Ping")
            print(f"Model '{model}' reachable (test reply: {str(reply.body).strip()[:10]!r})")
            return model
        except Exception as exc:  # the provider error text can echo part of the key, so print only the error type
            print(f"Model '{model}' not usable ({type(exc).__name__}); trying the next candidate...")
    raise RuntimeError("No candidate model is reachable. Check that the API key is valid and MODEL_CANDIDATES "
                       "contains a model your account can use.")


MODEL = await pick_model()


def make_config() -> OpenAIConfig:
    """One config per agent. Change MODEL above to switch every agent at once."""
    return OpenAIConfig(model=MODEL, api_key=API_KEY)

## 4. Step 1 — User Proxy Agent
Collects the user's inputs (**current salary** and a **summary of the current investment portfolio**), validates them with explicit rules (no LLM), and **initiates the chat with the Group Chat Manager**.

In [ ]:
from dataclasses import dataclass, asdict

VALID_CURRENCY_LEN = 3


@dataclass(frozen=True)
class UserInputs:
    salary: float
    currency: str
    portfolio_summary: str


def validate_inputs(salary, currency, portfolio_summary) -> UserInputs:
    """Deterministic validation: reject anything outside sane ranges or formats."""
    try:
        salary = float(str(salary).replace(",", ""))
    except ValueError:
        raise ValueError(f"salary must be a number, got {salary!r}") from None
    if not 1_000 <= salary <= 100_000_000:
        raise ValueError(f"salary must be between 1,000 and 100,000,000, got {salary}")
    currency = currency.strip().upper()
    if len(currency) != VALID_CURRENCY_LEN or not currency.isalpha():
        raise ValueError(f"currency must be a 3-letter code such as USD, got {currency!r}")
    portfolio_summary = " ".join(portfolio_summary.split())
    if not 10 <= len(portfolio_summary) <= 2000:
        raise ValueError("portfolio_summary must be between 10 and 2000 characters")
    return UserInputs(salary, currency, portfolio_summary)


class UserProxyAgent:
    """Collects user data (no LLM) and initiates the group chat."""

    name = "User_Proxy_Agent"

    def __init__(self, interactive: bool, defaults: dict):
        self.interactive = interactive
        self.defaults = defaults

    def _ask(self, label: str, default) -> str:
        raw = input(f"{label} [{default}]: ").strip() if self.interactive else ""
        return raw or str(default)

    def collect(self) -> UserInputs:
        """Collect salary + portfolio summary, re-prompting up to 3 times on invalid values."""
        print(f"[{self.name}] Collecting user data (press Enter to accept a default)")
        for attempt in range(3):
            try:
                inputs = validate_inputs(
                    self._ask("Current annual salary", self.defaults["salary"]),
                    self._ask("Currency (3-letter code)", self.defaults["currency"]),
                    self._ask("Summary of current investment portfolio", self.defaults["portfolio_summary"]),
                )
                print(f"[{self.name}] Collected: {asdict(inputs)}")
                return inputs
            except ValueError as exc:
                if not self.interactive or attempt == 2:
                    raise
                print(f"  Invalid input: {exc}. Please try again.\n")

    async def initiate_chat(self, manager, message: str):
        """Step 1: hand the user's intention + data to the Group Chat Manager, which runs the group chat."""
        return await manager.run(sender=self, message=message)


# Self-test: valid input passes, bad input must fail (should-fail cases).
assert validate_inputs("95,000", "usd", "Fixed deposits: 40,000; SIPs: 25,000") == UserInputs(95000.0, "USD", "Fixed deposits: 40,000; SIPs: 25,000")
VALIDATION_SHOULD_FAIL_OK = True
for bad in [("abc", "USD", "Fixed deposits: 40,000"), (0, "USD", "Fixed deposits: 40,000"), (95000, "DOLLARS", "Fixed deposits: 40,000"), (95000, "USD", "   ")]:
    try:
        validate_inputs(*bad)
        VALIDATION_SHOULD_FAIL_OK = False
        print("FAIL: accepted bad input", bad)
    except ValueError:
        pass
print("User Proxy Agent defined; validation self-test:", "PASS" if VALIDATION_SHOULD_FAIL_OK else "FAIL")

## 5. Steps 3, 5, 6 — Portfolio Analysis, Growth, Value and Investment Advisor Agents
Each agent gets a clear role through its `prompt`. The Portfolio Analysis Agent must end with a machine-readable `CATEGORY: Growth` or `CATEGORY: Value` line, which is the signal StateFlow uses to route the conversation.

In [ ]:
PORTFOLIO_AGENT = "Portfolio_Analysis_Agent"
GROWTH_AGENT = "Growth_Investment_Agent"
VALUE_AGENT = "Value_Investment_Agent"
ADVISOR_AGENT = "Investment_Advisor_Agent"

SHARED_RULES = ("You are taking part in a group chat managed by a Group Chat Manager. You see the full transcript. "
                "Reply only with your own contribution in Markdown, do not speak for other agents, and keep it under 250 words. "
                "This is general educational information, not licensed financial advice.")

portfolio_analysis_agent = Agent(
    PORTFOLIO_AGENT,
    prompt=f"""You are the Portfolio Analysis Agent. {SHARED_RULES}
Task: read the user's current salary and investment portfolio summary from the transcript, then:
1. Summarize the existing portfolio: list each holding, estimate the total, and describe the asset mix and risk level.
2. Determine the investment category to pursue next:
   - Growth: the user has a stable income and a long horizon, and the portfolio is dominated by low-risk or fixed-income
     assets, so there is room to add higher-growth exposure.
   - Value: the portfolio is already heavily in equities or illiquid assets, the user has near-term liabilities, or
     capital preservation and steady income matter more.
3. Give a 2-3 sentence justification.
End your reply with exactly one final line in this format and nothing after it:
CATEGORY: Growth
or
CATEGORY: Value""",
    config=make_config(),
)

growth_investment_agent = Agent(
    GROWTH_AGENT,
    prompt=f"""You are the Growth Investment Agent. {SHARED_RULES}
Task: using the Portfolio Analysis Agent's summary, recommend 4-5 specific high-growth investment options
(for example broad equity index funds, growth-oriented funds, sector or thematic funds, small-cap exposure)
suited to the user's salary and existing holdings. For each option give: the option, a suggested allocation (% of new
investments), the growth rationale, and the main risk. Start with the heading '### Growth investment recommendations'.""",
    config=make_config(),
)

value_investment_agent = Agent(
    VALUE_AGENT,
    prompt=f"""You are the Value Investment Agent. {SHARED_RULES}
Task: using the Portfolio Analysis Agent's summary, recommend 4-5 specific stable, long-term value investment options
(for example dividend or value index funds, investment-grade bonds, fixed deposits or treasury products, balanced funds)
suited to the user's salary and existing holdings. For each option give: the option, a suggested allocation (% of new
investments), the stability rationale, and the main risk. Start with the heading '### Value investment recommendations'.""",
    config=make_config(),
)

investment_advisor_agent = Agent(
    ADVISOR_AGENT,
    prompt=f"""You are the Investment Advisor Agent. {SHARED_RULES.replace('under 250 words', 'under 450 words')}
Task: compile a personalized financial report for the user from the group chat. Use exactly these Markdown headings, in order:
## Portfolio Analysis   (current holdings, total, asset mix, from the Portfolio Analysis Agent)
## Investment Category   (Growth or Value, and why)
## Recommended Investments   (the recommendations from the Growth or Value Investment Agent, as a table: Option | Allocation | Rationale | Risk)
## Action Plan   (3-5 concrete next steps with amounts based on the user's salary)
## Disclaimer   (one sentence: educational, not licensed financial advice)""",
    config=make_config(),
)

for agent in (portfolio_analysis_agent, growth_investment_agent, value_investment_agent, investment_advisor_agent):
    print("Agent ready:", agent.name)

## 6. Step 4 — StateFlow workflow management
StateFlow is a **deterministic state machine**: given the last speaker and the group chat so far, it returns who speaks next (or `None` to end the chat). This is the same `state_transition(last_speaker, groupchat)` signature that AG2 Classic passes as `speaker_selection_method`. The Growth/Value branch is decided by parsing the Portfolio Analysis Agent's `CATEGORY:` line (exact matching, not another LLM call).

In [ ]:
import re
from types import SimpleNamespace

CATEGORY_RE = re.compile(r"CATEGORY:\s*(Growth|Value)\b", re.IGNORECASE)


def parse_category(text: str):
    """Return 'Growth' or 'Value' from the LAST 'CATEGORY: ...' line, or None if the agent did not provide one."""
    matches = CATEGORY_RE.findall(text or "")
    return matches[-1].capitalize() if matches else None


def state_transition(last_speaker, groupchat):
    """StateFlow: User Proxy -> Portfolio Analysis -> (Growth | Value) -> Investment Advisor -> end."""
    if last_speaker is user_proxy:
        return portfolio_analysis_agent
    if last_speaker is portfolio_analysis_agent:
        category = parse_category(groupchat.messages[-1]["content"])
        if category is None:  # conservative fallback so the workflow still completes
            print("[StateFlow] No CATEGORY line found in the analysis; defaulting to Value (conservative).")
            category = "Value"
        print(f"[StateFlow] Portfolio category = {category}")
        return growth_investment_agent if category == "Growth" else value_investment_agent
    if last_speaker in (growth_investment_agent, value_investment_agent):
        return investment_advisor_agent
    if last_speaker is investment_advisor_agent:
        return None  # conversation complete
    raise ValueError(f"StateFlow has no transition for speaker {getattr(last_speaker, 'name', last_speaker)!r}")


# Self-test of the state machine without any LLM: both branches, the fallback, and the terminal state.
user_proxy = UserProxyAgent(interactive=False, defaults=DEFAULT_INPUTS)  # the real run re-creates it with INTERACTIVE_INPUT
fake = lambda content: SimpleNamespace(messages=[{"name": PORTFOLIO_AGENT, "content": content}])
STATEFLOW_SELFTEST_OK = all([
    state_transition(user_proxy, fake("")) is portfolio_analysis_agent,
    state_transition(portfolio_analysis_agent, fake("...\nCATEGORY: Growth")) is growth_investment_agent,
    state_transition(portfolio_analysis_agent, fake("...\ncategory: value")) is value_investment_agent,
    state_transition(portfolio_analysis_agent, fake("no category line at all")) is value_investment_agent,  # fallback
    state_transition(growth_investment_agent, fake("")) is investment_advisor_agent,
    state_transition(value_investment_agent, fake("")) is investment_advisor_agent,
    state_transition(investment_advisor_agent, fake("")) is None,
])
print("StateFlow self-test:", "PASS" if STATEFLOW_SELFTEST_OK else "FAIL")

## 7. Step 2 — Group Chat and Group Chat Manager
`GroupChat` holds the shared `messages` that every agent sees. `GroupChatManager` runs the rounds: it asks StateFlow who speaks next, sends that agent the **full transcript**, appends the reply, and stops when StateFlow returns `None` or `max_round` is reached.

In [ ]:
class GroupChat:
    """Shared conversation: every speaker receives the full transcript (collaboration)."""

    def __init__(self, agents: list, max_round: int, speaker_selection_method):
        self.agents = agents
        self.max_round = max_round
        self.speaker_selection_method = speaker_selection_method
        self.messages: list[dict] = []

    def transcript(self) -> str:
        return "\n\n".join(f"[{m['name']}]:\n{m['content']}" for m in self.messages)


class GroupChatManager:
    """Manages the group chat and directs the conversation flow using the StateFlow function."""

    name = "Group_Chat_Manager"

    def __init__(self, groupchat: GroupChat):
        self.groupchat = groupchat
        self.speaker_order: list[str] = []
        self.ended_by_stateflow = False

    async def run(self, sender, message: str) -> list[dict]:
        gc = self.groupchat
        gc.messages.clear()
        self.speaker_order.clear()
        self.ended_by_stateflow = False
        line = "-" * 80
        gc.messages.append({"name": sender.name, "content": message, "visible_messages": 0})
        print(f"{line}\n{sender.name} -> {self.name}\n{line}\n{message}\n")

        last_speaker = sender
        for round_no in range(1, gc.max_round + 1):
            speaker = gc.speaker_selection_method(last_speaker, gc)
            if speaker is None:
                self.ended_by_stateflow = True
                print(f"[{self.name}] StateFlow returned None after {round_no - 1} rounds: conversation complete.")
                break
            visible = len(gc.messages)
            print(f"[{self.name}] Round {round_no}: next speaker -> {speaker.name} (sees {visible} messages)")
            prompt = f"Group chat transcript so far:\n\n{gc.transcript()}\n\nIt is now your turn, {speaker.name}."
            reply = await speaker.ask(prompt)
            gc.messages.append({"name": speaker.name, "content": reply.body or "", "visible_messages": visible})
            self.speaker_order.append(speaker.name)
            print(f"{line}\n{speaker.name}\n{line}\n{reply.body}\n")
            last_speaker = speaker
        else:
            print(f"[{self.name}] max_round={gc.max_round} reached; stopping.")
        return gc.messages


group_chat = GroupChat(
    agents=[portfolio_analysis_agent, growth_investment_agent, value_investment_agent, investment_advisor_agent],
    max_round=MAX_ROUND,
    speaker_selection_method=state_transition,
)
group_chat_manager = GroupChatManager(groupchat=group_chat)
print("Group chat ready with agents:", [a.name for a in group_chat.agents])

## 8. Run the conversation
Step 1: the User Proxy Agent collects the data and initiates the chat with the Group Chat Manager, informing it of the user's intention to manage their investments. The manager and StateFlow then drive the rest.

In [ ]:
user_proxy = UserProxyAgent(interactive=INTERACTIVE_INPUT, defaults=DEFAULT_INPUTS)
user_inputs = user_proxy.collect()

initial_message = (
    "I want to manage my investments and get a personalized financial plan.\n"
    f"- Current annual salary: {user_inputs.salary:,.0f} {user_inputs.currency}\n"
    f"- Current investment portfolio: {user_inputs.portfolio_summary}"
)

messages = await user_proxy.initiate_chat(group_chat_manager, message=initial_message)
print("Speaker order:", " -> ".join(group_chat_manager.speaker_order))

## 9. Final output — the Investment Advisor Agent's personalized financial report

In [ ]:
from IPython.display import Markdown, display

advisor_messages = [m for m in messages if m["name"] == ADVISOR_AGENT]
final_report = advisor_messages[-1]["content"] if advisor_messages else ""
display(Markdown(f"# Personalized Financial Report\n\n*Prepared by {ADVISOR_AGENT} for a user earning "
                 f"{user_inputs.salary:,.0f} {user_inputs.currency} per year*\n\n{final_report}"))

## 10. Rubric verification
Automated check of the rubric table at the top. Every row should print `PASS`.

In [ ]:
order = group_chat_manager.speaker_order
analysis = next((m["content"] for m in messages if m["name"] == PORTFOLIO_AGENT), "")
category = parse_category(analysis)
expected_specialist = {"Growth": GROWTH_AGENT, "Value": VALUE_AGENT}.get(category, VALUE_AGENT)
other_specialist = VALUE_AGENT if expected_specialist == GROWTH_AGENT else GROWTH_AGENT
specialist_text = next((m["content"] for m in messages if m["name"] == expected_specialist), "")
required_sections = ("## Portfolio Analysis", "## Investment Category", "## Recommended Investments", "## Action Plan", "## Disclaimer")
text = lambda s: (s or "").lower()

checks = [
    ("R1  User Proxy collected salary + portfolio summary and rejects bad input",
     isinstance(user_inputs, UserInputs) and VALIDATION_SHOULD_FAIL_OK),
    ("R2  User Proxy initiated the chat with the Group Chat Manager (intention to manage investments)",
     messages and messages[0]["name"] == user_proxy.name and "manage my investments" in text(messages[0]["content"])),
    ("R3  Group Chat Manager called the Portfolio Analysis Agent first", order[:1] == [PORTFOLIO_AGENT]),
    ("R4  Portfolio Analysis Agent summarized the portfolio and determined Growth or Value",
     category in ("Growth", "Value") and any(w in text(analysis) for w in ("portfolio", "holding", "deposit"))),
    ("R5  StateFlow routed to the matching specialist only (other specialist never spoke)",
     order[1:2] == [expected_specialist] and other_specialist not in order),
    ("R6  StateFlow state machine unit-tested (both branches, fallback, terminal state)", STATEFLOW_SELFTEST_OK),
    ("R7  Specialist agent provided investment recommendations",
     "recommend" in text(specialist_text) and len(specialist_text) > 200),
    ("R8  Investment Advisor Agent spoke last and compiled the report", order[-1:] == [ADVISOR_AGENT] and len(order) == 3),
    ("R9  Conversation ended through StateFlow within max_round",
     group_chat_manager.ended_by_stateflow and len(order) <= MAX_ROUND),
    ("R10 Every speaker saw all previous messages (group-chat collaboration)",
     all(m["visible_messages"] == i for i, m in enumerate(messages))),
    ("R11 Final report has portfolio analysis + investment suggestions (all required sections)",
     all(s in final_report for s in required_sections) and category.lower() in text(final_report)),
    ("R12 Investment Advisor Agent's output was displayed as Markdown", bool(final_report.strip())),
]

for name, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\n{sum(bool(ok) for _, ok in checks)}/{len(checks)} checks passed")